# 02 — Indexing & Slicing

**Dataset**: `sklearn.datasets.load_digits` — 1797 handwritten digits, 8×8 grayscale images, 10 classes.  
**Goal**: Slice specific images, crop regions, apply boolean masks, and use fancy indexing — in both NumPy and PyTorch.

In [3]:
# ── Shared Setup ────────────────────────────────────────────────────────────
from sklearn.datasets import load_digits
import numpy as np
import torch
import matplotlib.pyplot as plt

digits = load_digits()

np_images = digits.images.astype(np.float32)   # (1797, 8, 8)
np_labels = digits.target.astype(np.int64)     # (1797,)

pt_images = torch.tensor(np_images)
pt_labels = torch.tensor(np_labels)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'images: {np_images.shape} | labels: {np_labels.shape}')

images: (1797, 8, 8) | labels: (1797,)


---
## P1 — Basic Slicing: Extract a Batch

Grab the first 32 images and their labels (a mini-batch).

In [4]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
np_batch  = np_images[:32]    # (32, 8, 8)
np_batch_y = np_labels[:32]   # (32,)
print('batch:', np_batch.shape, '| labels:', np_batch_y.shape)

batch: (32, 8, 8) | labels: (32,)


#### Drill — `slice_batch`
Practice the core operation before using it in the problem above.

In [5]:
t = torch.arange(10)
# DRILL: fetch elements 2,3,4
x = t[2:5]
assert x.numel() == 3

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def slice_batch(imgs, labs, size=32):
    """
    Returns the first `size` images and their labels.
    """
    t_batch   = images   # imgs[:size]
    t_batch_y = ...
    return t_batch, t_batch_y

t_batch, t_batch_y = slice_batch(pt_images, pt_labels)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert t_batch.shape == (32, 8, 8)
assert t_batch_y.shape == (32,)
assert np.allclose(np_batch, t_batch.numpy())
print('P1 assertions passed ✓')

---
## P2 — Cropping: Extract Center 4×4 Region

From each 8×8 image, crop the center 4×4 patch (rows 2:6, cols 2:6).

In [ ]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
np_cropped = np_images[:, 2:6, 2:6]   # (1797, 4, 4)
print('cropped:', np_cropped.shape)

# Visualize crop
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i in range(5):
    axes[0, i].imshow(np_images[i], cmap='gray'); axes[0, i].set_title('full 8×8')
    axes[1, i].imshow(np_cropped[i], cmap='gray'); axes[1, i].set_title('center 4×4')
    axes[0, i].axis('off'); axes[1, i].axis('off')
plt.tight_layout(); plt.show()

#### Drill — `crop_center`
Practice the core operation before using it in the problem above.

In [ ]:
t = torch.zeros(4, 4)
# DRILL: crop 2x2 center
center = t[1:3, 1:3]
assert center.shape == (2, 2)

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def crop_center(imgs):
    """
    Crop center 4×4 from all (N, 8, 8) images.
    """
    # Same slicing syntax as NumPy
    t_cropped = ...
    return t_cropped

t_cropped = crop_center(pt_images)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert t_cropped.shape == (1797, 4, 4)
assert np.allclose(np_cropped, t_cropped.numpy())
print('P2 assertions passed ✓')

---
## P3 — Boolean Masking: Select One Class

Extract all images of digit `3` using a boolean mask on the labels.

In [ ]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
CLASS = 3
np_mask_3   = np_labels == CLASS          # (1797,) bool
np_class_3  = np_images[np_mask_3]        # (N_3, 8, 8)
np_count_3  = np_mask_3.sum()
print(f'digit {CLASS}: {np_count_3} samples, shape {np_class_3.shape}')

#### Drill — `select_class`
Practice the core operation before using it in the problem above.

In [ ]:
t = torch.tensor([0, 1, 0, 1])
t_labs = torch.tensor([0, 1, 0, 1])
# DRILL: fetch elements where label is 1
x = t[t_labs == 1]
assert x.sum() == 2

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def select_class(imgs, labs, cls=3):
    """
    Filter images by class label.

    Returns:
        t_mask    : (N,) bool tensor
        t_class   : (N_cls, 8, 8) filtered images
        t_count   : int — number of samples in the class
    """
    t_mask  = ...
    t_class = ...
    t_count = ...

    return t_mask, t_class, t_count

t_mask_3, t_class_3, t_count_3 = select_class(pt_images, pt_labels, CLASS)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert t_count_3 == np_count_3
assert t_class_3.shape == np_class_3.shape
assert np.allclose(np_class_3, t_class_3.numpy())
print(f'P3 assertions passed ✓  ({t_count_3} images of digit {CLASS})')

---
## P4 — Pixel Threshold Masking

Given the first image, create a binary mask for pixels > 8 (mid-range for [0,16]).
Then use `np.where` / `torch.where` to replace dark pixels with 0 and bright pixels with 16.

In [ ]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
THRESHOLD = 8.0
sample_np  = np_images[0]                                # (8, 8)
np_bright  = sample_np[sample_np > THRESHOLD]            # 1-D array of bright pixels
np_binary  = np.where(sample_np > THRESHOLD, 16.0, 0.0) # binarised image

print(f'bright pixels: {np_bright.shape[0]} / 64')
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(sample_np, cmap='gray'); axes[0].set_title('Original')
axes[1].imshow(np_binary, cmap='gray'); axes[1].set_title('Binarised')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

#### Drill — `threshold_ops`
Practice the core operation before using it in the problem above.

In [ ]:
t = torch.tensor([1, 10, 2, 20])
# DRILL: fetch elements >= 10
big = t[t >= 10]
assert big.numel() == 2

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def threshold_ops(img_t, threshold=8.0):
    """
    Args:  img_t — (8, 8) tensor

    Returns:
        t_bright  : 1-D tensor of pixels above threshold
        t_binary  : (8, 8) tensor — 16 where bright, 0 where dark
    """
    # Step 1: boolean mask then index
    t_bright = ...

    # Step 2: torch.where — element-wise conditional
    # NumPy equivalent: np.where(X > threshold, 16.0, 0.0)
    t_binary = ...

    return t_bright, t_binary

t_bright, t_binary = threshold_ops(pt_images[0])

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert t_bright.shape == np_bright.shape
assert np.allclose(np_bright, t_bright.numpy())
assert np.allclose(np_binary, t_binary.numpy())
print('P4 assertions passed ✓')

---
## P5 — Fancy Indexing: Gather Specific Samples

Select images at indices `[0, 42, 100, 999, 1796]` — like picking specific exam questions to study.

In [ ]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
PICK = [0, 42, 100, 999, 1796]
np_picked = np_images[PICK]           # (5, 8, 8)
np_picked_y = np_labels[PICK]         # (5,)
print('picked shapes:', np_picked.shape, np_picked_y.shape)

#### Drill — `fancy_index`
Practice the core operation before using it in the problem above.

In [ ]:
t = torch.arange(10, 20)
# DRILL: get indices [0, -1] via a list
x = t[[0, -1]]
assert x[0] == 10 and x[1] == 19

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def fancy_index(imgs, labs, indices):
    """
    Gather specific images by index.

    Returns:
        t_picked   : (len(indices), 8, 8)
        t_picked_y : (len(indices),)
    """
    # Option A: plain Python-list indexing (works like NumPy)
    t_picked   = ...
    t_picked_y = ...

    # Option B (PyTorch-specific): torch.index_select
    #   idx_tensor = torch.tensor(indices)
    #   t_picked = torch.index_select(imgs, dim=0, index=idx_tensor)

    return t_picked, t_picked_y

t_picked, t_picked_y = fancy_index(pt_images, pt_labels, PICK)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert t_picked.shape == (5, 8, 8)
assert np.allclose(np_picked, t_picked.numpy())
assert np.array_equal(np_picked_y, t_picked_y.numpy())
print('P5 assertions passed ✓')

---
## P6 — masked_fill (PyTorch-only)

Use `masked_fill` to blank out (set to 0) all pixels ≤ threshold in the first image.  
This is the same pattern used for attention masking in Transformers.

#### Drill — `masked_fill_demo`
Practice the core operation before using it in the problem above.

In [ ]:
t = torch.tensor([1., 5., 2., 10.])
# DRILL: fill elements >= 5 with 0
t.masked_fill_(t >= 5, 0.)
assert t.sum() == 3

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def masked_fill_demo(img_t, threshold=4.0):
    """
    Args:  img_t — (8, 8) tensor

    Returns:
        t_filled : (8, 8) tensor with dark pixels replaced by 0
    """
    dark_mask = ...
    t_filled  = ...
    return t_filled

t_filled = masked_fill_demo(pt_images[0])

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert t_filled.shape == (8, 8)
# All pixels ≤ 4 should be exactly 0
assert (t_filled[pt_images[0] <= 4.0] == 0).all()
# Pixels > 4 should be unchanged
assert (t_filled[pt_images[0] > 4.0] == pt_images[0][pt_images[0] > 4.0]).all()
print('P6 assertions passed ✓')